In [2]:
# goal: prototype skeleton transformer
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4
import re

import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path

In [3]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker
from src.nano_maker.modules.nano_printer.smiles_handler import smiles_fingerprint
from src.database.dataset import RadialDataset

In [4]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
data_folder = Path("notebook_database")
RADIAL_SEQUENCES = pd.read_pickle(data_folder/"radial_seq_df.pkl")
MOLECULAR_FINGERPRINTS = pd.read_pickle(data_folder/"molfp_df.pkl")
TRAIN_POINTER = pd.read_parquet(data_folder/"training_pointers.parquet")
TEST_POINTER = pd.read_parquet(data_folder/"test_pointers.parquet")

In [5]:
print(len(TRAIN_POINTER))
print(len(TEST_POINTER))

14120350
3630380


In [6]:
print(MOLECULAR_FINGERPRINTS.head(3))
print(RADIAL_SEQUENCES.head(3))

sample_fp = MOLECULAR_FINGERPRINTS["map4_fp"].iloc[0]
print(len(sample_fp), sample_fp[0].dtype)

                                          smiles_str  \
0  C[C@@H](N1CC[C@](C)(C1=O)c1ccc(OCc2cc(C)nc3ccc...   
1  Cc1ccc(CNc2cc(OC[C@H]3C[C@@H]3c3ccc4CCCc4n3)nn...   
2  C[C@@H](COc1ccnc2CCC[C@@H](C)c12)C[C@H]1Cc2cc3...   

                                             map4_fp  
0  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
1  [tensor(0.), tensor(1.), tensor(0.), tensor(0....  
2  [tensor(0.), tensor(1.), tensor(1.), tensor(1....  
  PDB_ID                                    radial_sequence
0   5UEY  [[[I], [61, 56, 32]], [[K], [68, 53, 41]], [[M...
1   6FID  [[[Y], [46, 67, 40]], [[D], [67, 59, 52]], [[H...
2   1WCC  [[[L], [35, 47, 36]], [[K], [65, 61, 44]], [[V...
1024 torch.float32


In [7]:
sample_train_pointer = TRAIN_POINTER.sample(1000000)
sample_test_pointer = TEST_POINTER.sample(100000)

sample_test_pointer

,SMILES,PDB_HIT,WINDOW_IDX
471731,CCN(Cc1csc(C(=O)Nc2c(OC)cc(Cl)cc2C(=O)Nc2ccc(C...,3TU7,47
1021087,COC(=O)C(CSc1ccc(Br)cc1)n1c(=O)n2CC=CC(C(=O)NC...,6ZUU,28
1105765,COCCCN(CCOC)C[C@H](O)[C@@H]1N2CC[C@H]2CN2C[C@@...,6U67,27
2195838,Cc1cc2nc(-c3nccs3)n(-c3cc4nc(N)nc(N)c4cc3Cl)c2...,4LAH,33
2625914,Cn1ncc(NC(=O)c2csc(n2)-c2c(F)cc(cc2F)C(C)(F)F)...,5KZI,35
...,...,...,...
644333,CCc1cc(O)c(F)cc1-c1ccc2c(n[nH]c2c1)-c1nc2CC(N(...,3E64,21
1335647,COc1cc2c(C(=O)N(COC3=C(C(=O)OC3)c3ccccc3)S2(=O...,5A8X,48
1297494,COc1cc(cc(OC)c1OC)C1CC(=NN1c1ccc(cc1)S(N)(=O)=...,6UFD,19
1496487,COc1ccc(cc1)-n1nc(C#N)c2CCN(C(=O)c12)c1ccc(cc1...,2J95,5


In [8]:
# RadialSeeker parameters used -> record these - might design a config file later
radial_resolution = 100
intrashell_resolution = 100
max_angstroms = 33

batch_size = 360  # how many data X X Ys we want to pass at a time
block_size = 10  # coordinate context
max_iters = 500
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 3
dropout = 0.2
coord_num = 101
map4_res = 1024

# Dataset parameters
training_dataset = RadialDataset(pointer=sample_train_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size)
# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

torch.manual_seed(311104)

In [9]:
class SkeletonModel(nn.Module):
    def __init__(self, n_embd, n_head, block_size, map4_res, radial_resolution):
        super().__init__()
        self.block_size = block_size
        self.map4_res = map4_res
        self.radial_resolution = radial_resolution

        self.c_project = nn.Linear(3, n_embd)  # feed coordinates here
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.mol_project = nn.Linear(map4_res, n_embd)

        self.stack = Stack(n_embd, n_head)

        # OUTPUT HEAD -> outputs coordinates
        self.c_head = nn.Linear(n_embd, 3)


    def forward(self, coord_hist, map4_enc, targets=None):
        """
        coord_hist will be [n_batch, block_size, 3]
        targets is [n_batch, 1, 3]
        map4_enc -> unsure how I'm going to encode this
        """
        B, T, C = coord_hist.shape
        coordinate_emb = self.c_project(coord_hist.float() / self.radial_resolution)
        pos_emb = self.pos_emb(torch.arange(T))
        x = coordinate_emb + pos_emb

        mol_emb = self.mol_project(map4_enc.float()).unsqueeze(1)

        x = self.stack(x, mol_emb)

        output_coords = self.c_head(x[:, -1, :])

        loss = None
        if targets is not None:
            loss = F.mse_loss(output_coords, (targets.float() / self.radial_resolution))

        return output_coords, loss

    def generate(self, map4_enc, max_steps=130):
        # largest protein pocket in dataset was 107
        coord_context = torch.zeros(1, block_size, 3)
        coord_out = []

        for _ in range(max_steps):
            next_coord, _ = self.forward(coord_context, map4_enc)
            coord_out.append(next_coord)
            coord_context = torch.cat([coord_context[:, 1:, :], next_coord.unsqueeze(1)], dim=1)



        return coord_out


In [10]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute affinities
        attn_wts = q @ k.transpose(-2, -1) * C**-0.5
        attn_wts = F.softmax(attn_wts, dim=-1)
        attn_wts = self.dropout(attn_wts)

        # weighed aggregation
        values = self.value(x)
        output = attn_wts @ values
        return output


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out

In [11]:
class CrossAttentionHead(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.2):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coordinate_emb, molecular_emb):
        q = self.query(coordinate_emb)
        k = self.key(molecular_emb)
        v = self.value(molecular_emb)

        head_size = q.shape[-1]

        c_attn = q @ k.transpose(-2, -1) * head_size**-0.5
        c_attn = F.softmax(c_attn, dim=-1)
        c_attn = self.dropout(c_attn)

        return c_attn @ v


class MultiHeadCrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.2):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([CrossAttentionHead(n_embd,
                                                       head_size,
                                                       ) for _ in range(n_head)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, coord_emb, mol_emb):
        out = torch.cat([h(coord_emb, mol_emb) for h in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out

In [12]:
class FeedForwardNet(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
class Stack(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.satt_head = MultiHeadAttention(n_embd, n_head)
        self.ffwd = FeedForwardNet(n_embd)
        self.catt_head = MultiHeadCrossAttention(n_embd, n_head)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, mol_emb):
        x = x + self.satt_head(self.ln1(x))
        x = x + self.catt_head(self.ln2(x), mol_emb)
        x = x + self.ffwd(self.ln3(x))
        return x

In [14]:
# TODO: build small MLP for MAP4 encoding

In [15]:
model = SkeletonModel(n_embd=256, n_head=4, block_size=10, map4_res=1024, radial_resolution=100)

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

SkeletonModel(
  (c_project): Linear(in_features=3, out_features=256, bias=True)
  (pos_emb): Embedding(10, 256)
  (mol_project): Linear(in_features=1024, out_features=256, bias=True)
  (stack): Stack(
    (satt_head): MultiHeadAttention(
      (heads): ModuleList(
        (0-3): 4 x Head(
          (key): Linear(in_features=256, out_features=64, bias=False)
          (query): Linear(in_features=256, out_features=64, bias=False)
          (value): Linear(in_features=256, out_features=64, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
      (projection): Linear(in_features=256, out_features=256, bias=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (ffwd): FeedForwardNet(
      (net): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.2, inplace=False)
      )
    )
    (catt_head): MultiHeadCrossAtten

In [17]:
for epoch in range(5):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, radial_X, radial_Y = batch

        optimizer.zero_grad()
        pred, loss = model(radial_X, map4_fp, radial_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    print(f"Epoch {epoch + 1} finished. Avg loss: {
    total_loss/len(loader):.6f}")

Epoch 1 | Batch 1 | Loss: 0.716667


KeyboardInterrupt: 

In [18]:
# Dataset parameters
test_set = RadialDataset(pointer=sample_test_pointer,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size)
# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=4,
    shuffle=False,
    drop_last=True
)

with torch.no_grad():


SyntaxError: incomplete input (1879252640.py, line 16)

In [19]:
from src.nano_maker.modules.nano_printer.smiles_handler import smiles_fingerprint

sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"
sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
model.generate(sample_map4)

<>:3: SyntaxWarning: invalid escape sequence '\c'
<>:3: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_2794145/19181139.py:3: SyntaxWarning: invalid escape sequence '\c'
  sample_smiles = "CC/C(=C(\c1ccc(cc1)O)/c2ccc(cc2)OCCN(C)C)/c3ccccc3"


[tensor([[0.5088, 0.4231, 0.4759]], grad_fn=<AddmmBackward0>),
 tensor([[0.4830, 0.5068, 0.4319]], grad_fn=<AddmmBackward0>),
 tensor([[0.4334, 0.4581, 0.5496]], grad_fn=<AddmmBackward0>),
 tensor([[0.4050, 0.3827, 0.5043]], grad_fn=<AddmmBackward0>),
 tensor([[0.3808, 0.3070, 0.5516]], grad_fn=<AddmmBackward0>),
 tensor([[0.4185, 0.5144, 0.4600]], grad_fn=<AddmmBackward0>),
 tensor([[0.4718, 0.4125, 0.5638]], grad_fn=<AddmmBackward0>),
 tensor([[0.4745, 0.3601, 0.5068]], grad_fn=<AddmmBackward0>),
 tensor([[0.4565, 0.4234, 0.4533]], grad_fn=<AddmmBackward0>),
 tensor([[0.4946, 0.5224, 0.5700]], grad_fn=<AddmmBackward0>),
 tensor([[0.4021, 0.3447, 0.5938]], grad_fn=<AddmmBackward0>),
 tensor([[0.5967, 0.5332, 0.5429]], grad_fn=<AddmmBackward0>),
 tensor([[0.3731, 0.4989, 0.4714]], grad_fn=<AddmmBackward0>),
 tensor([[0.4381, 0.3948, 0.5365]], grad_fn=<AddmmBackward0>),
 tensor([[0.4510, 0.4821, 0.4563]], grad_fn=<AddmmBackward0>),
 tensor([[0.5019, 0.4802, 0.5231]], grad_fn=<AddmmBackw